# Ablation 2026-09-01 — LDM + Encoder + Temp+SDF
**Run:** `abl_ldm_enc_sdf`  |  **W&B:** `1_Sep_2026_ldm_enc_sdf`  
**Model:** LDMModel (4x VAE latent space, DDPM sampler)  
**Encoder:** RRDB warm-start (implicit conditioning)  |  **Fields:** `temperature` + `sdfliqlabel`


In [ ]:
RUN_NAME='abl_ldm_enc_sdf'; WANDB_RUN_NAME='1_Sep_2026_ldm_enc_sdf'
LDM_RUN_DIR='/scratch/ngng/runs/abl_ldm_enc_sdf'
ENC_RUN_DIR='/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/enc_sdf'
VAE_RUN_DIR='/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/vae_sdf'
DATA_ROOT='/trace/group/forgelab/ngng/multifield/data_fields'
EVAL_OUT_DIR='/trace/group/forgelab/ngng/multifield/eval_results/ablation_20260901/abl_ldm_enc_sdf'
FIELD_NAMES=['temperature','sdfliqlabel']; N_STEPS=3; DOWNSCALE_METHOD='direct'; NORMALIZE='standardize'
TIMESTEPS=1000; SCHEDULE='linear'; ENCODING=True; CONDITIONING='implicit'; DEVICE='cuda'
BATCH_INDEX=0; SAMPLE_INDEX=0; BATCH_SIZE=4; T_LIQ=1700.0; LIQ_THR=0.5
MELT_THRESHOLD=1900.0; ANALYSIS_CH=0; ANALYSIS_MAX_BATCH=None

In [ ]:
%matplotlib inline
import os,sys,time; from pathlib import Path
import numpy as np,torch,pandas as pd; import matplotlib.pyplot as plt
from torch.utils.data import DataLoader; from scipy.ndimage import gaussian_filter as _gf
from IPython.display import display as _ipy_display
plt.show=lambda *a,**kw:[_ipy_display(plt.figure(n)) for n in plt.get_fignums()] or plt.close('all')
if not torch.cuda.is_available() and DEVICE=='cuda': DEVICE='cpu'; print('CPU fallback')
def find_root(s=Path.cwd()):
    for p in [s,*s.parents]:
        if (p/'setup.py').exists() and (p/'diffusionsr').exists(): return p
    raise RuntimeError('no root')
PROJECT_ROOT=find_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
print(f'root={PROJECT_ROOT} device={DEVICE}')

In [ ]:
from diffusionsr.datasets.dataset import SimulationXZDataset
from diffusionsr.analysis.analysis_functions import get_profile, load_encoder
from scipy.spatial import KDTree; from scipy.ndimage import binary_erosion
def as_numpy(x): return x.detach().cpu().numpy() if isinstance(x,torch.Tensor) else np.asarray(x)
def mae_rmse(p,g): p,g=np.asarray(p).ravel(),np.asarray(g).ravel(); return {'MAE':float(np.mean(np.abs(p-g))),'RMSE':float(np.sqrt(np.mean((p-g)**2)))}
def boundary_pixels(m): return np.argwhere(m&~binary_erosion(m,structure=np.ones((3,3))))
def chamfer_dist(a,b):
    ba,bb=boundary_pixels(a),boundary_pixels(b)
    if len(ba)==0 or len(bb)==0: return float('nan')
    return float((KDTree(bb).query(ba)[0].mean()+KDTree(ba).query(bb)[0].mean())/2)
def consistency_metrics(bt,bl):
    i,u=(bt&bl).sum(),(bt|bl).sum(); return (i/u if u>0 else float('nan')),float(np.mean((bt.astype(float)-bl.astype(float))**2)),chamfer_dist(bt,bl)
fn=FIELD_NAMES; has_sdf='sdfliqlabel' in fn; has_T='temperature' in fn; has_liq='liqlabel' in fn or has_sdf
def _liq_mask(p): return p[fn.index('sdfliqlabel')]<0 if has_sdf else p[fn.index('liqlabel')]>LIQ_THR
def _mk_contour(ax,p,sigma=1.5):
    if not(has_T and has_liq): return
    try: ax.contour(_gf(_liq_mask(p).T.astype(float),sigma),levels=[0.5],colors=['white'],linewidths=[1.],origin='lower',alpha=0.85)
    except: pass

In [ ]:
from diffusionsr.runners.train_ldm import LDMModel
kw=dict(downscale_method=DOWNSCALE_METHOD,root_folder=DATA_ROOT,normalize=NORMALIZE,n_steps=N_STEPS,field_names=FIELD_NAMES)
train_ds,dev_ds,test_ds=(SimulationXZDataset(split=s,**kw) for s in ['train','dev','test'])
print(f'Fields:{train_ds.field_names} HR:{train_ds.img_shape} {train_ds.factor}x')
model=LDMModel(vae_folder=VAE_RUN_DIR,results_folder=LDM_RUN_DIR,lr_encoder_folder=ENC_RUN_DIR,
    train_dataset=train_ds,dev_dataset=dev_ds,test_dataset=test_ds,
    timesteps=TIMESTEPS,conditioning=CONDITIONING,encoding=ENCODING,schedule=SCHEDULE,device=DEVICE,enc_output=False)
model.load_saved_model()
print(f'LDMModel loaded: {LDM_RUN_DIR}')
print(f'VAE: {VAE_RUN_DIR}  Encoder: {ENC_RUN_DIR}')

In [ ]:
loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False)
for i,batch in enumerate(loader):
    if i==BATCH_INDEX: break
_,hr_s,lr_s,ul_s=[x[SAMPLE_INDEX:SAMPLE_INDEX+1] for x in batch[:4]]
x_e=model.compute_x_e(lr_s,ul_s)
with torch.no_grad():
    samps=model.batch_sample(dataset=test_ds,batch=hr_s.to(DEVICE),x_e=x_e,sampler='DDPM')
    mu_hr,_=model.vae.encode(hr_s.to(DEVICE).float()); vae_r=model.vae.decode(mu_hr).cpu().numpy()
pred_phys=test_ds.unscale_data(samps[-1].cpu().numpy()[0],input_type='hr')
hr_phys=test_ds.unscale_data(as_numpy(hr_s[0]),input_type='hr')
lr_phys=test_ds.unscale_data(as_numpy(lr_s[0]),input_type='lr')
up_phys=test_ds.unscale_data(as_numpy(ul_s[0]),input_type='upscaled_lr')
vae_phys=test_ds.unscale_data(vae_r[0],input_type='hr')
enc_phys=test_ds.unscale_data(as_numpy(model.lr_enc(ul_s.to(DEVICE)).cpu()[0]),input_type='hr')
fig,axes=plt.subplots(1,6,figsize=(26,4),dpi=120)
for ax,(ttl,data,phys) in zip(axes,[('LR',lr_phys[0],lr_phys),('Upscaled',up_phys[0],up_phys),('CNN',enc_phys[0],enc_phys),('VAE recon',vae_phys[0],vae_phys),('LDM',pred_phys[0],pred_phys),('GT',hr_phys[0],hr_phys)]):
    ax.imshow(data.T,origin='lower',cmap='jet',vmin=293,vmax=5000,aspect='auto'); _mk_contour(ax,phys); ax.set_title(ttl,fontsize=9); ax.axis('off')
plt.suptitle(f'{RUN_NAME}',fontsize=10); plt.tight_layout(); plt.show()
print(f'VAE recon MAE: {mae_rmse(vae_phys,hr_phys)["MAE"]:.4f}')

In [ ]:
from diffusionsr.runners.plot_training_curves import collect_curves
all_c=[('Enc:'+l,v) for l,v in collect_curves(Path(ENC_RUN_DIR))]+collect_curves(Path(LDM_RUN_DIR))
if all_c:
    fig,ax=plt.subplots(figsize=(9,4),dpi=120)
    [ax.plot(v,ls='--' if 'val' in l.lower() else '-',label=l,lw=1.5) for l,v in all_c]
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title(f'{RUN_NAME} — Curves'); ax.legend(fontsize=8); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Full test-set stats (DDPM sampler)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,drop_last=False)
maes,rmses,mp_errs,kh_errs,vc_maes,vae_maes=[],[],[],[],[],[]
iou_list,cham_list,iou_gt_list,cham_gt_list=[],[],[],[]
t0=time.perf_counter(); n_samples=0
for i,batch in enumerate(test_loader):
    if ANALYSIS_MAX_BATCH is not None and i>=ANALYSIS_MAX_BATCH: break
    _,hr_b,lr_b,ul_b=batch[:4]
    xe=model.compute_x_e(lr_b,ul_b)
    with torch.no_grad():
        samps=model.batch_sample(dataset=test_ds,batch=hr_b.to(DEVICE),x_e=xe,sampler='DDPM')
        mu,_=model.vae.encode(hr_b.to(DEVICE).float()); vr=model.vae.decode(mu).cpu().numpy()
    n_samples+=hr_b.shape[0]
    for s in range(hr_b.shape[0]):
        p=test_ds.unscale_data(samps[-1].cpu().numpy()[s],input_type='hr')
        g=test_ds.unscale_data(as_numpy(hr_b[s]),input_type='hr')
        vp=test_ds.unscale_data(vr[s],input_type='hr')
        m=mae_rmse(p[ANALYSIS_CH],g[ANALYSIS_CH]); maes.append(m['MAE']); rmses.append(m['RMSE'])
        vae_maes.append(mae_rmse(vp[ANALYSIS_CH],g[ANALYSIS_CH])['MAE'])
        try:
            pmp,pkh=get_profile(p[ANALYSIS_CH:ANALYSIS_CH+1]); gmp,gkh=get_profile(g[ANALYSIS_CH:ANALYSIS_CH+1])
            mp_errs.append(float(np.mean(np.abs(pmp-gmp)))); kh_errs.append(float(np.mean(np.abs(pkh-gkh))))
        except: pass
        vc_maes.append(float(np.mean(np.abs((p[ANALYSIS_CH]>MELT_THRESHOLD).astype(float)-(g[ANALYSIS_CH]>MELT_THRESHOLD).astype(float)))))
        if has_T and has_liq:
            iou,_,cham=consistency_metrics(p[fn.index('temperature')]>T_LIQ,_liq_mask(p))
            ioug,_,chamg=consistency_metrics(g[fn.index('temperature')]>T_LIQ,_liq_mask(g))
            iou_list.append(iou); cham_list.append(cham); iou_gt_list.append(ioug); cham_gt_list.append(chamg)
elapsed=time.perf_counter()-t0
print(f'n={len(maes)} DDPM (LDM)  MAE={np.nanmean(maes):.4f}\u00b1{np.nanstd(maes):.4f}')
print(f'  VAE recon MAE={np.nanmean(vae_maes):.4f} (upper bound from compression)')
if mp_errs: print(f'  MP-MAE={np.nanmean(mp_errs):.2f}px  KH-MAE={np.nanmean(kh_errs):.2f}px')
if iou_list: print(f'  IOU pred={np.nanmean(iou_list):.4f}  GT={np.nanmean(iou_gt_list):.4f}')
print(f'  Speed:{n_samples/elapsed:.2f} s/s')

In [ ]:
# Multi-sample grid (5 cols: LR|Up|CNN|LDM|GT)
N_GRID=5; _gl=DataLoader(test_ds,batch_size=N_GRID,shuffle=False,drop_last=True); _gb=next(iter(_gl))
_res_g,_hr_g,_lr_g,_ul_g=_gb[:4]; _xe_g=model.compute_x_e(_lr_g,_ul_g)
with torch.no_grad():
    _sg=model.batch_sample(dataset=test_ds,batch=_hr_g.to(DEVICE),x_e=_xe_g,sampler='DDPM')
    _ec_g=model.lr_enc(_ul_g.to(DEVICE)).cpu()
_COLS=['LR Input','Upscaled LR','CNN Encoder','LDM','GT']
fig,axes=plt.subplots(N_GRID,5,figsize=(22,3.5*N_GRID),dpi=90)
if N_GRID==1: axes=axes[np.newaxis]
for r in range(N_GRID):
    _p=test_ds.unscale_data(_sg[-1].cpu().numpy()[r],input_type='hr'); _g=test_ds.unscale_data(as_numpy(_hr_g[r]),input_type='hr')
    _l=test_ds.unscale_data(as_numpy(_lr_g[r]),input_type='lr'); _u=test_ds.unscale_data(as_numpy(_ul_g[r]),input_type='upscaled_lr')
    _e=test_ds.unscale_data(as_numpy(_ec_g[r]),input_type='hr')
    for c,(data,phys) in enumerate(zip([_l[0],_u[0],_e[0],_p[0],_g[0]],[_l,_u,_e,_p,_g])):
        ax=axes[r,c]; ax.imshow(data.T,origin='lower',cmap='jet',vmin=293,vmax=5000,aspect='auto'); _mk_contour(ax,phys); ax.axis('off')
        if r==0: ax.set_title(_COLS[c],fontsize=9,fontweight='bold')
plt.suptitle(f'{RUN_NAME} — grid (DDPM)',fontsize=10); plt.tight_layout(); plt.show()

# DDIM skip ablation
_ABL_SKIPS=[1,5,10,20,50]; _ABL_N=4; _ab_mae={s:[] for s in _ABL_SKIPS}; _ab_mp={s:[] for s in _ABL_SKIPS}
for _ai,_ab in enumerate(DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False)):
    if _ai>=_ABL_N: break
    _,_ahr,_alr,_aul=_ab[:4]; _axe=model.compute_x_e(_alr,_aul)
    for _sk in _ABL_SKIPS:
        with torch.no_grad(): _as=model.batch_sample(dataset=test_ds,batch=_ahr.to(DEVICE),x_e=_axe,sampler='DDIM',skip=_sk)
        for _s in range(_ahr.shape[0]):
            _pp=test_ds.unscale_data(_as[-1].cpu().numpy()[_s],input_type='hr')
            _gg=test_ds.unscale_data(as_numpy(_ahr[_s]),input_type='hr')
            _ab_mae[_sk].append(mae_rmse(_pp[ANALYSIS_CH],_gg[ANALYSIS_CH])['MAE'])
            try: _pmp,_=get_profile(_pp[ANALYSIS_CH:ANALYSIS_CH+1]); _gmp,_=get_profile(_gg[ANALYSIS_CH:ANALYSIS_CH+1]); _ab_mp[_sk].append(float(np.mean(np.abs(_pmp-_gmp))))
            except: pass
_nfe=[TIMESTEPS//_sk for _sk in _ABL_SKIPS]
_mae_v=[np.nanmean(_ab_mae[s]) for s in _ABL_SKIPS]; _mp_v=[np.nanmean(_ab_mp[s]) if _ab_mp[s] else float('nan') for s in _ABL_SKIPS]
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,5),dpi=120)
ax1.plot(_nfe,_mae_v,'o-',color='steelblue',lw=2,ms=8); ax1.set_xlabel('NFE (DDIM steps)'); ax1.set_ylabel('MAE'); ax1.set_title('DDIM Ablation — Field MAE'); ax1.grid(True,alpha=0.3)
ax2.plot(_nfe,_mp_v,'s-',color='teal',lw=2,ms=8); ax2.set_xlabel('NFE (DDIM steps)'); ax2.set_ylabel('MP-MAE (px)'); ax2.set_title('DDIM Ablation — MP Depth'); ax2.grid(True,alpha=0.3)
plt.suptitle(f'{RUN_NAME} — DDIM step ablation (skip={_ABL_SKIPS})',fontsize=11); plt.tight_layout(); plt.show()

In [ ]:
import os; os.makedirs(EVAL_OUT_DIR,exist_ok=True)
summary={'run_name':RUN_NAME,'wandb_run':WANDB_RUN_NAME,'model':'LDM','encoding':ENCODING,'conditioning':CONDITIONING,
         'fields':str(FIELD_NAMES),'sampler':'DDPM','n_test':len(maes),
         'mae_mean':float(np.nanmean(maes)),'mae_std':float(np.nanstd(maes)),
         'rmse_mean':float(np.nanmean(rmses)),'rmse_std':float(np.nanstd(rmses)),
         'vae_mae_mean':float(np.nanmean(vae_maes)) if vae_maes else float('nan'),
         'mp_mae_mean':float(np.nanmean(mp_errs)) if mp_errs else float('nan'),
         'kh_mae_mean':float(np.nanmean(kh_errs)) if kh_errs else float('nan'),
         'vc_mae_mean':float(np.nanmean(vc_maes)) if vc_maes else float('nan'),
         'iou_mean':float(np.nanmean(iou_list)) if iou_list else float('nan')}
pd.DataFrame([summary]).to_csv(f'{EVAL_OUT_DIR}/metrics_summary.csv',index=False)
print(f'Saved -> {EVAL_OUT_DIR}/metrics_summary.csv'); pd.DataFrame([summary]).T